# 🦀 Day 4: `thiserror` and `anyhow` Crates

These crates make error handling ergonomic and professional!

---

## 📦 `thiserror` — For Libraries

Generates `Display`, `Error`, and `From` implementations from derive macros.

```toml
[dependencies]
thiserror = "1"
```

```rust
use thiserror::Error;

#[derive(Debug, Error)]
enum DataError {
    #[error("file not found: {path}")]
    NotFound { path: String },

    #[error("parse error on line {line}: {message}")]
    ParseError { line: usize, message: String },

    #[error("IO error")]
    Io(#[from] std::io::Error),  // Auto-generates From<io::Error>

    #[error("invalid format: {0}")]
    InvalidFormat(String),
}
```

The `#[error(...)]` attribute generates your `Display` impl!
The `#[from]` attribute generates `From<E>` for `?` operator!

## 📦 `anyhow` — For Applications

```toml
[dependencies]
anyhow = "1"
```

```rust
use anyhow::{Context, Result, bail, ensure};

fn process(path: &str) -> Result<String> {  // anyhow::Result
    let content = std::fs::read_to_string(path)
        .context(format!("Failed to read {}", path))?;  // Adds context!

    ensure!(!content.is_empty(), "File {} is empty", path);  // Like assert but returns Err

    let number: i32 = content.trim().parse()
        .context("Failed to parse number")?;

    if number < 0 {
        bail!("Number must be positive, got {}", number);  // Return Err immediately
    }

    Ok(format!("Processed: {}", number))
}
```

In [ ]:
// Let's simulate thiserror behavior manually
// (since we can't easily load crates in EvCxR)

use std::fmt;

#[derive(Debug)]
enum ConfigError {
    MissingField(String),
    InvalidValue { field: String, expected: String, got: String },
    IoError(std::io::Error),
}

impl fmt::Display for ConfigError {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        match self {
            ConfigError::MissingField(name) =>
                write!(f, "missing required field: {}", name),
            ConfigError::InvalidValue { field, expected, got } =>
                write!(f, "invalid value for '{}': expected {}, got {}", field, expected, got),
            ConfigError::IoError(e) =>
                write!(f, "IO error: {}", e),
        }
    }
}

impl std::error::Error for ConfigError {}
impl From<std::io::Error> for ConfigError {
    fn from(e: std::io::Error) -> Self { ConfigError::IoError(e) }
}

// With thiserror, all the above would be:
// #[derive(Debug, Error)]
// enum ConfigError {
//     #[error("missing required field: {0}")]
//     MissingField(String),
//     #[error("invalid value for '{field}': expected {expected}, got {got}")]
//     InvalidValue { field: String, expected: String, got: String },
//     #[error("IO error")]
//     IoError(#[from] std::io::Error),
// }

println!("ConfigError: {}", ConfigError::MissingField("port".into()));
println!("ConfigError: {}", ConfigError::InvalidValue {
    field: "port".into(), expected: "1-65535".into(), got: "99999".into()
});

## 🤔 When to Use Which?

| | `thiserror` | `anyhow` |
|---|---|---|
| **For** | Libraries | Applications |
| **Error types** | Custom enum | `anyhow::Error` (any error) |
| **Context** | Manual (in Display) | `.context()` method |
| **Use when** | Callers need to match specific errors | You just want to report errors |
| **Performance** | Zero overhead | Small allocation per error |

In [ ]:
// Simulating anyhow-style context
use std::fs;

fn read_config() -> Result<String, Box<dyn std::error::Error>> {
    // In real code with anyhow:
    // let content = fs::read_to_string("config.toml")
    //     .context("Failed to read config file")?;

    let content = fs::read_to_string("config.toml")
        .map_err(|e| format!("Failed to read config file: {}", e))?;
    Ok(content)
}

match read_config() {
    Ok(c) => println!("Config: {}", c),
    Err(e) => println!("Error: {}", e),  // Shows context + original error
}

## 🏋️ Exercises

In [ ]:
// Exercise: Create error types for a user registration system:
// - EmptyField(field_name)
// - TooShort { field, min_length, actual_length }
// - InvalidEmail(String)
// - PasswordMismatch
// Write validation functions that use these errors

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **`thiserror`**: generates Display, Error, From from derive macros
2. **`anyhow`**: wraps any error with `.context()`, bail!, ensure!
3. Libraries → `thiserror` (structured errors callers can match on)
4. Applications → `anyhow` (just report errors clearly)
5. Both work with `?` operator seamlessly

---
**Tomorrow:** panic! vs recoverable errors! 💥